## Load the two datasets
Clean the them (thoroughly)
Measure at least 1 data engineering metric in the data cleaning process (ie dropped rows)
MVP: Output the cleaned files as a local csv.
Stretch:

Output the cleaned files into the local SSMS (use SQLAlchemy); create a new DB for this.
Refactor your code into modular functions.

## Imports

In [2]:
import pandas as pd
import datetime as dt
from sqlalchemy import create_engine

In [73]:
today = dt.date.today().strftime('%d/%m/%Y')
print(today)

02/06/2026


## Load dataset

In [74]:
df_library_books = pd.read_csv(r'C:\Users\Admin\Desktop\20260601-DEM5\data\03_Library Systembook.csv')
df_library_books.head(50)

,Id,Books,Book checkout,Book Returned,Days allowed to borrow,Customer ID
0,1.0,Catcher in the Rye,"""20/02/2023""",25/02/2023,2 weeks,1.0
1,2.0,Lord of the rings the two towers,"""24/03/2023""",21/03/2023,2 weeks,2.0
2,3.0,Lord of the rings the return of the kind,"""29/03/2023""",25/03/2023,2 weeks,3.0
3,4.0,The hobbit,"""02/04/2023""",25/03/2023,2 weeks,4.0
4,5.0,Dune,"""02/04/2023""",25/03/2023,2 weeks,5.0
5,6.0,Little Women,"""02/04/2023""",01/05/2023,2 weeks,1.0
6,7.0,IT,"""10/04/2063""",03/04/2023,2 weeks,6.0
7,8.0,Misery,"""15/04/2023""",03/04/2023,2 weeks,7.0
8,9.0,Catch 22,"""15/04/2023""",16/04/2023,2 weeks,7.0
9,10.0,Animal Farm,"""20/04/2023""",24/04/2023,2 weeks,2.0


In [4]:
df_library_customers = pd.read_csv(r'C:\Users\Admin\Desktop\20260601-DEM5\app\data\03_Library SystemCustomers.csv')
df_library_customers.head(20)

,Customer ID,Customer Name
0,1.0,Jane Doe
1,2.0,John Smith
2,3.0,Dan Reeves
3,NaN,NaN
4,5.0,William Holden
5,6.0,Jaztyn Forest
6,7.0,Jackie Irving
7,8.0,Matthew Stirling
8,9.0,Emory Ted


## Clean Dataset

cleaning steps:

1) remove NaN rows
2) Change ID to int
3) Book checkout and book return to date
4) days allowed to borrow to int
5) customer id to int

In [76]:
## Functions

def convert_columns_to_int(df, column_name):
    '''
    converts column to integer
    '''
    df[column_name] = pd.to_numeric(df[column_name], errors='coerce').astype('Int64')

    return df


def drop_nulls(df):
    '''
    drop rows where all values are null
    '''
    df.dropna(how='all', inplace=True)

    return df 


def keep_values_only(df, column_name):
    df[column_name] = df[column_name].astype(str).str.extract(r'(\d+)', expand=False).astype('Int64')

    return df


def convert_to_dates(df, column_name, date_format):

    df[column_name] = pd.to_datetime(df[column_name], format=date_format, errors='coerce')

    return df 

def rename_columns(df, columns_mapping: dict):
    return df.rename(columns=columns_mapping)


In [77]:
def convert_and_validate_dates(df):
    date_columns = ["Book checkout", "Book Returned"]

    # Convert date columns
    for col in date_columns:
        df[col] = (
            df[col]
            .astype(str)
            .str.replace('"', '', regex=False)
        )

        df[col] = pd.to_datetime(
            df[col],
            format="%d/%m/%Y",
            errors="coerce"
        )

    # Invalid rows:
    # 1. Any date conversion failed (NaT)
    # 2. Book checkout is in the future
    invalid_rows = df[
        df["Book checkout"].isna() |
        df["Book Returned"].isna() |
        (df["Book checkout"] > today)
    ]

    return df, invalid_rows

In [78]:
## remove NA's
df_library_books = drop_nulls(df_library_books)

## Convert Id and Customer Id to integers
df_library_books = convert_columns_to_int(df_library_books, 'Id')
df_library_books = convert_columns_to_int(df_library_books, 'Customer ID')

## Convert book checkout and return to dates
df_library_books, invalid_df = convert_and_validate_dates(df_library_books)


## extract values from text
df_library_books = keep_values_only(df_library_books, 'Days allowed to borrow')

## rename columns
mapping = {'Id': 'id',
 'Books': 'books',
 'Book checkout': 'book_checkout',
 'Book Returned': 'book_returned',
 'Days allowed to borrow': 'weeks_allowed_to_borrow',
 'Customer ID': 'customer_id'
 }

df_library_books = rename_columns(df_library_books, mapping)

In [79]:
df_library_books.head(50)

,id,books,book_checkout,book_returned,weeks_allowed_to_borrow,customer_id
0,1,Catcher in the Rye,2023-02-20,2023-02-25,2,1
1,2,Lord of the rings the two towers,2023-03-24,2023-03-21,2,2
2,3,Lord of the rings the return of the kind,2023-03-29,2023-03-25,2,3
3,4,The hobbit,2023-04-02,2023-03-25,2,4
4,5,Dune,2023-04-02,2023-03-25,2,5
5,6,Little Women,2023-04-02,2023-05-01,2,1
6,7,IT,2063-04-10,2023-04-03,2,6
7,8,Misery,2023-04-15,2023-04-03,2,7
8,9,Catch 22,2023-04-15,2023-04-16,2,7
9,10,Animal Farm,2023-04-20,2023-04-24,2,2


In [80]:
invalid_df.head(5)

,Id,Books,Book checkout,Book Returned,Days allowed to borrow,Customer ID
6,7,IT,2063-04-10,2023-04-03,2 weeks,6
16,17,The Bloody Chamber,NaT,2023-06-04,2 weeks,3


In [ ]:
def save_df_to_sql(df, table_name, conn_string, if_exists='append'):

    engine = create_engine(conn_string)

    df.to_sql(
        table_name,
        con=engine,
        if_exists = if_exists,
        index = False
    )

    return True

In [ ]:
save_df_to_sql(df_library_books, 'library_books', 'sqlite:///DE5M5_Library.db')

True

In [88]:
def date_diff(df, start_col, end_col):

    date_difference = pd.to_datetime(df[end_col]) - pd.to_datetime(df[start_col])
    date_difference_days = date_difference.dt.days

    return date_difference_days


In [89]:
df_library_books['days_diff'] = date_diff(df_library_books, 'book_returned', 'book_checkout')
df_library_books.head(20)

,id,books,book_checkout,book_returned,weeks_allowed_to_borrow,customer_id,days_diff
0,1,Catcher in the Rye,2023-02-20,2023-02-25,2,1,-5.0
1,2,Lord of the rings the two towers,2023-03-24,2023-03-21,2,2,3.0
2,3,Lord of the rings the return of the kind,2023-03-29,2023-03-25,2,3,4.0
3,4,The hobbit,2023-04-02,2023-03-25,2,4,8.0
4,5,Dune,2023-04-02,2023-03-25,2,5,8.0
5,6,Little Women,2023-04-02,2023-05-01,2,1,-29.0
6,7,IT,2063-04-10,2023-04-03,2,6,14617.0
7,8,Misery,2023-04-15,2023-04-03,2,7,12.0
8,9,Catch 22,2023-04-15,2023-04-16,2,7,-1.0
9,10,Animal Farm,2023-04-20,2023-04-24,2,2,-4.0
